# 平衡器和变压器设计

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf
from skrf.circuit import Circuit
from skrf.constants import c

rf.stylely()

本教程将介绍一些无源射频电路，例如使用传输线的铁氧体负载平衡器和变压器，所有这些都将在 scikit-rf 环境中进行。我们已在 Windows Anaconda、Android Pydroid 3 和 iOS Juno 上测试了这些电路。## 平衡-不平衡电路平衡-不平衡电路是将平衡电路转换为不平衡电路，反之亦然。变压器，顾名思义，是将一个射频阻抗转换为另一个射频阻抗。平衡器和变压器都可以使用传输线或集总元件构建。平衡器和变压器被广泛使用，但并非仅用于射频匹配电路、组合器、分配器、放大器、混频器。有时，平衡器和变压器也用于天线设计、天线模拟器和/或天线调谐器（也称为耦合器）。<img src="Balun_Transformer_Designs/Simple_Balun2.png"/>图 1：同轴平衡器的理想原理图

如图 1 所示，描绘了一个同轴平衡器，它由一个 4 端口（外导体悬空）同轴电缆和一个电阻组成。在该电路中，电阻用于模拟同轴电缆周围铁氧体套筒（或珠子）的效果。该电阻的作用是抑制外导体外表面上的电流流动，从而使同轴电缆的行为与感兴趣带宽内的双绞线类似。在某些情况下，可以使用集总电感器代替电阻，以模拟套筒中铁氧体材料的频率相关行为。电阻或电感器的实际值是设计频率范围和铁氧体材料特性的函数。通常，电阻或电感器的值是从实验室的实验测量中获得的。我们在这里使用 333 欧姆作为一个合理的值，仅用于演示 scikit-rf 的用法。实际上，在 scikit-rf 中，一个 4 端口传输线正是按照下图中的节点图方式实现的：<img src="Balun_Transformer_Designs/Nodal_Connections_Simple_Balun.png" />图 2：节点连接

以图 2 为参考，我们可以按以下方式创建一个传输线平衡器。首先，我们在传输线上添加一些插入损耗，形式为 0.1 欧姆电阻。然后，我们将所有元件（变压器、两条 25 欧姆的线路和铁氧体电阻）连接在一起，以创建平衡器。总端口为 50 欧姆单端端口。差分输出端口均为 25 欧姆。

In [ ]:
# need this ideal s4p file to create a coax with a floating outer conductor
xformer = r"Balun_Transformer_Designs/Ideal_4port_xformer_hi_res.s4p"
xformer_ntwk = rf.Network(xformer)
xformer_ntwk1 = xformer_ntwk["0.001-2ghz"]  # band center is thus 1 GHz

freq = xformer_ntwk1.frequency
z0_ports = 25

beta = freq.w / c
line_branch = rf.media.DefinedGammaZ0(frequency=freq, z0=z0_ports, gamma=0+beta*1j)

# d is 75 mm in length, which is quarterwave at 1 GHz in air
#d = line_branch.theta_2_d(90, deg=True, bc=True)
branch1 = line_branch.line(90, unit="deg", name="branch1")

lossy_line = line_branch.resistor(0.1, name="res")

该射频电路是使用 scikit-rf 的 [Circuit] 功能构建的：

In [ ]:
port1 = Circuit.Port(freq, name="port1", z0=25)
port2 = Circuit.Port(freq, name="port2", z0=25)

# creating a 25-Ohm 2-port transmission line, with realistic insertion loss
# insertion loss is equivalent to a 0.1-Ohm series resistor
connections = [
    [(port1, 0), (lossy_line, 0)],
    [(lossy_line, 1), (branch1, 0)],
    [(branch1, 1), (port2, 0)],
]

C = Circuit(connections)

line1 = C.network
line1.name = "top line"
line2 = C.network
line2.name = "bottom line"

line = rf.media.DefinedGammaZ0(frequency=freq, z0=50)
ferrite_resistor = line.resistor(333, name="res")

port1a = Circuit.Port(frequency=freq, name="port1a", z0=50)
port2a = Circuit.Port(frequency=freq, name="port2a", z0=25)
port3a = Circuit.Port(frequency=freq, name="port3a", z0=25)
ground = Circuit.Ground(frequency=freq, name="ground", z0=50)

# creating a 4-port floating coax by feeding two 25-Ohm lines (twisted pair of wires)
# with an ideal transformer, to ensure the differential nature of the twisted pair, as per Fig.2
# the 333-Ohm resistor models the loading of the ferrite sleeve,
# upon the outer conductor of the 4-port coax line
conn = [
    [(port1a, 0), (xformer_ntwk1, 0)],
    [(xformer_ntwk1, 1), (line1, 0)],
    [(line1, 1), (port2a, 0)],
    [(ground, 0), (xformer_ntwk1, 2), (ferrite_resistor, 0)],
    [(xformer_ntwk1, 3), (line2, 0)],
    [(line2, 1), (ferrite_resistor, 1), (port3a, 0)],
]

C1 = Circuit(conn)
balun = C1.network
balun.name = "ideal balun"

让我们看一下这个理想平衡器的散射参数：

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

balun.plot_s_db(ax=ax1, m=0, n=0, lw=2)  # S11
balun.plot_s_db(ax=ax1, m=1, n=1, lw=2)  # S22
ax1.set_ylim(-60, 0)

balun.plot_s_db(ax=ax2, m=1, n=0, lw=2)  # S21
ax2.set_ylim(-4, 0)

fig.suptitle("Ideal 50-Ohm balun")

如上图所示，总端口回波损耗 S11（蓝色曲线，上图）为 -35dB。S22（红色曲线，上图）为 -6dB，这表示差分对中一个端子的典型回波损耗。S21（蓝色曲线，下图）为 -3dB，如预期，这表明进入总端口的能量均匀地分配到两个差分端口。上述平衡器的设计示例是在 scikit-rf 版本 1.3 中实现的（也与早期版本兼容）。对于 scikit-rf 版本 1.4，以下脚本使用新创建的 Media 元素，称为 [line_floating](https://scikit-rf.readthedocs.io/en/latest/api/media/generated/skrf.media.Media.line_floating.html#skrf.media.Media.line_floating)，以简化电路。

In [ ]:
freq1 = rf.Frequency(start=0.01, stop=2, unit=('GHz'), npoints=2001)
beta = freq1.w / c

line1 = rf.media.DefinedGammaZ0(frequency=freq1, z0=50, gamma=0+beta*1j)

ferrite_resistor = line1.resistor(333, name='res') # model ferrite sleeve as a parallel 333 Ohm resistor
Rp_circuit = ferrite_resistor

# define a floating 4-port transmission line, using a new element in ver 1.4 of Scikit-rf
line4p = line1.line_floating(90, unit='deg', z0=50, name='line4p')
port1 = Circuit.Port(frequency=freq1, name='port1', z0=50)
port2 = Circuit.Port(frequency=freq1, name='port2', z0=25)
port3 = Circuit.Port(frequency=freq1, name='port3', z0=25)
ground = Circuit.Ground(frequency=freq1, name='ground', z0=50)

connections = [[(port1, 0), (line4p, 0)],
    [(port2, 0), (line4p, 1)],
    [(port3, 0), (line4p, 3), (Rp_circuit, 0)],
    [(line4p, 2), (Rp_circuit, 1), (ground, 0)]]

circuit = Circuit(connections)
balun_circuit = circuit.network
balun_circuit.name = 'ideal balun circuit'

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

balun_circuit.plot_s_db(ax=ax1, m=0, n=0, lw=2)  # S11
balun_circuit.plot_s_db(ax=ax1, m=1, n=1, lw=2)  # S22
ax1.set_ylim(-60, 0)

balun_circuit.plot_s_db(ax=ax2, m=1, n=0, lw=2)  # S21
ax2.set_ylim(-4, 0)

fig.suptitle("Ideal 50-Ohm balun")

## Guanella 变压器接下来要探讨的电路是一个 Guanella 变压器，它是一个 4:1 的阻抗变压器。在这个例子中，一个 100 欧姆的阻抗通过两个 50 欧姆的传输线被变换为 25 欧姆。这个例子是在 scikit-rf 1.4 版本中实现的。<img src="Balun_Transformer_Designs/Guanella_transformer_1.png"/>图 3：理想的 Guanella 变压器

以图 3 为参考，可以创建一个 Guanella 变压器。请注意，在电路的左侧，两条传输线以射频串联方式连接，形成一个 100 欧姆端口。在另一端，右侧以并联方式连接，形成一个 25 欧姆端口。此外，在本示例中，铁氧体套筒被建模为电感器。

In [ ]:
freq = rf.Frequency(start=0.01, stop=2, unit=('GHz'), npoints=2001)
beta = freq.w/c

line = rf.media.DefinedGammaZ0(frequency=freq, z0=50, gamma=0+beta*1j)
ferrite_inductor = line.inductor(100e-9, name='ind') # model ferrite sleeve as a parallel 100 nH inductor

# define a floating 4-port transmission line
line4p_a = line.line_floating(90, unit='deg', z0=50, name='line4p_a')
line4p_b = line.line_floating(90, unit='deg', z0=50, name='line4p_b')

port1 = Circuit.Port(frequency=freq, name='port1', z0=100)
port2 = Circuit.Port(frequency=freq, name='port2', z0=25)
ground1 = Circuit.Ground(frequency=freq, name='ground1', z0=50)
ground2 = Circuit.Ground(frequency=freq, name='ground2', z0=50)

connections = [[(port1, 0), (line4p_a, 0)],
    [(port2, 0), (line4p_a, 1), (line4p_b, 1)],
    [(line4p_a, 2), (line4p_b, 0), (ferrite_inductor, 0)],
    [(line4p_a, 3), (ferrite_inductor, 1), (ground1, 0), (line4p_b, 3)],
    [(line4p_b, 2), (ground2, 0)]]

circuit = Circuit(connections)
transf_circuit = circuit.network
transf_circuit.name = 'ideal Guanella transformer circuit'

transf_circuit.plot_s_db(m=0, n=0, lw=2)
transf_circuit.plot_s_db(m=1, n=0, lw=2)
transf_circuit.plot_s_db(m=1, n=1, lw=2)

plt.ylim(-50,10)

在这个关内拉变压器示例中，铁氧体套筒被建模为一个并联在同轴线外导体的感性元件。 如图所示的散射参数图中，该变压器在 200 到 1000 MHz 之间工作良好。 在 200 MHz 以下，铁氧体套筒感性元件提供的阻抗不足以有效地“扼制”同轴线。 在 1000 MHz 时，该电路产生一个尖锐的谐振，这限制了最高频率为 1 GHz。

## 马尚平衡器我们在此讨论的最后一个电路是三轴[马尚平衡器](https://www.microwaves101.com/encyclopedias/marchand-balun)，它不使用任何铁氧体材料。在这个电路中，两个三轴部分串联，平衡输出从中间引出。图 4 显示了马尚平衡器的示意图，与原始马尚论文中的描述相符。<img src="Balun_Transformer_Designs/Annotated_Marchand.png" />图 4 原始马尚平衡器示意图

如图 4 所示，同轴线的横截面具有一个内导体，该内导体由中间的环形外导体屏蔽。反过来，环形最外层外导体包围着中间导体。图 5 显示了一个模拟同轴 Marchand 平衡器的电路。使用了两个嵌套的同轴线来模拟一个同轴结构。<img src="Balun_Transformer_Designs/Marchand_Balun.png"/>图 5：理想的 Marchand 平衡器电路

如图 5 所示，同轴电缆 CX1 和 CX2 构成第一个三轴线。最内层的同轴电缆 CX1 的阻抗为 50 欧姆。CX1 的外导体成为 120 欧姆 CX2 的内导体。第二个三轴线由 10 欧姆的 CX3 和 120 欧姆的 CX4 构成。与前一个三轴线类似，CX3 的外导体成为 CX4 的内导体。图 5 中的理想变压器用于说明 Marchand 平衡器的输出确实是差分信号。

In [ ]:
# ideal transformer to convert differential ports to 50-Ohms single-ended
xformer = r"Balun_Transformer_Designs/Ideal_4port_xformer_hi_res.s4p"
xformer_ntwk = rf.Network(xformer)
xformer_ntwk1a = xformer_ntwk["0.001-2ghz"]

freq = xformer_ntwk1a.frequency
beta = freq.w/c

line = rf.media.DefinedGammaZ0(frequency=freq, z0=50, gamma=0+beta*1j)
C1 = line.capacitor(0.106e-12, name='cap')

# define a floating 4-port transmission line
line4p_a1 = line.line_floating(90, unit='deg', z0=50, name='line4p_a1')
line4p_b1 = line.line_floating(90, unit='deg', z0=120, name='line4p_b1')
line4p_c1 = line.line_floating(90, unit='deg', z0=10, name='line4p_c1')
line4p_d1 = line.line_floating(90, unit='deg', z0=120, name='line4p_d1')

port1a = Circuit.Port(frequency=freq, name='port1', z0=50)
port2a = Circuit.Port(frequency=freq, name='port2', z0=50)
ground1a = Circuit.Ground(frequency=freq, name='ground1', z0=50)
ground2a = Circuit.Ground(frequency=freq, name='ground2', z0=50)

connections = [[(port1a, 0), (line4p_a1, 0)],
    [(ground1a, 0), (line4p_a1, 2), (line4p_b1, 0), (line4p_b1, 2), (line4p_b1, 3)],
    [(line4p_a1, 1), (line4p_c1, 0)],
    [(line4p_a1, 3), (line4p_b1, 1), (xformer_ntwk1a, 0)],
    [(line4p_c1, 2), (xformer_ntwk1a, 2), (line4p_d1, 0)],
    [(line4p_d1, 1), (line4p_c1, 3), (line4p_d1, 2), (line4p_d1, 3), (C1, 1), (xformer_ntwk1a, 3), (ground2a, 0)],
    [(line4p_c1, 1), (C1, 0)],
    [(xformer_ntwk1a, 1), (port2a, 0)]]

circuit1 = Circuit(connections)
Marchand_circuit = circuit1.network
Marchand_circuit.name = 'Marchand circuit'

Marchand_circuit.plot_s_db(m=0, n=0, lw=2)
Marchand_circuit.plot_s_db(m=1, n=0, lw=2)
Marchand_circuit.plot_s_db(m=1, n=1, lw=2)

plt.ylim(-50,10)

在上述 Marchand 平衡器的 S 参数图中，该电路的带宽为 250 到 1600 MHz（-20dB 回波损耗带宽）。 这种多八度频段的性能是在不使用任何铁氧体套筒的情况下实现的。 这里的权衡是平衡器的结构复杂性。通过这三个简单的例子，可以在 scikit-rf 中构建许多其他有趣的传输线电路。 此外，这些电路可以在 iPhone (Juno) 上、Android 平板电脑 (Pydroid 3) 上或笔记本电脑 (Anaconda) 上构建。 这种易于访问的方式使得设计理念能够快速且准确地记录并进行测试，无论何时何地产生这些想法。 可能性和潜力是无限的。